# Knowledge base — ingestion, pgvector, and retrieval quality

The corpus is what makes the advisory *hyper-local* rather than generic. This notebook
covers the three things worth checking:

1. **Ingestion** — front matter becomes node metadata, which is what makes filtered
   retrieval possible.
2. **Backends** — pgvector when Postgres is up, an on-disk store otherwise. Identical
   interface either way, so nothing downstream knows the difference.
3. **Retrieval quality** — does the right bulletin come back, and does the severity tier in
   the query actually change what is retrieved?

Everything here is free: `bge-small-en-v1.5` runs locally on CPU with no API key and no rate
limit, and Postgres is the `pgvector/pgvector:pg16` container.

In [ ]:
from _bootstrap import setup

ROOT = setup()

from app.config import get_settings
from app.core.logging import configure_logging

configure_logging("WARNING")
settings = get_settings()

print(f"corpus     : {settings.knowledge_dir}")
print(f"embeddings : {settings.embedding_model} ({settings.embedding_dim} dims)")
print(f"database   : {settings.database_url or '(unset -> local file store)'}")
print(f"table      : {settings.pgvector_table}")
print(f"top_k/keep : {settings.retrieval_top_k}/{settings.retrieval_keep}")

## 1. The corpus

Authoring convention: Markdown with YAML front matter carrying `crop`, `disease`, `region`,
`doc_type` and `source`. `file_name`, `source`, `region` and `doc_type` are excluded from the
embedded text — a file path adds noise to the vector, not signal — but stay available as
metadata for filtering and citation.

The five seed documents are illustrative. Phase 2 replaces them with real extension
bulletins for the target region; the format does not change.

In [ ]:
from app.rag.ingest import load_documents

documents, skipped = load_documents(settings.knowledge_dir)
print(f"{len(documents)} documents" + (f", skipped: {skipped}" if skipped else "") + "\n")

for document in documents:
    metadata = document.metadata
    print(f"{metadata['file_name']:<42} {metadata['crop']:<10} {metadata['disease']:<14} "
          f"{metadata['doc_type']:<18} {len(document.text):>6,} chars")

## 2. Chunking

768-token chunks with 96 overlap. The size is chosen so a complete staged recommendation
("at 10-25 % severity, do X at rate Y") survives inside one chunk. Split it and the model
retrieves the rate without the severity condition it applies to — which is exactly the kind
of error that produces a dangerous recommendation.

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

from app.rag.ingest import CHUNK_OVERLAP, CHUNK_SIZE

splitter = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
nodes = splitter.get_nodes_from_documents(documents)

lengths = [len(node.get_content()) for node in nodes]
print(f"{len(nodes)} nodes | chars min {min(lengths):,} / mean {sum(lengths) // len(lengths):,} / max {max(lengths):,}\n")

per_document = {}
for node in nodes:
    key = node.metadata.get("file_name", "?")
    per_document[key] = per_document.get(key, 0) + 1
for name, count in sorted(per_document.items()):
    print(f"  {count:>2} nodes  {name}")

print("\n--- first node ---")
print(nodes[0].get_content()[:700])

## 3. Ingest

`prefer_pgvector=False` forces the on-disk store. Leave it `True` (the default) and it uses
pgvector when reachable and silently falls back otherwise — the behaviour the API relies on.

In [ ]:
from app.rag.ingest import ingest_knowledge

knowledge, report = ingest_knowledge(settings, prefer_pgvector=False, reset=True)
print(report)
print(f"\nbackend    : {knowledge.backend}")
print(f"embeddings : {knowledge.embedding.model_name} ({knowledge.embedding.dim} dims)"
      + ("  [DEGRADED: hash fallback]" if knowledge.embedding.degraded else ""))

## 4. Retrieval quality

Two questions worth answering with evidence rather than assertion:

**(a) Does the right document come back for a direct question?**

In [ ]:
QUERIES = [
    "late blight tomato treatment wet weather",
    "early blight concentric rings target spot lower leaves",
    "potato tuber storage after blight infection",
    "copper oxychloride rate organic certification",
    "what severity justifies a chemical spray",
]

retriever = knowledge.index.as_retriever(similarity_top_k=3)
for query in QUERIES:
    print(f"\nQ: {query}")
    for node in retriever.retrieve(query):
        metadata = node.node.metadata
        print(f"   {node.score:.3f}  {metadata.get('file_name', '?'):<40} "
              f"{metadata.get('crop', '')}/{metadata.get('disease', '')}")

**(b) Does the severity tier in the query change what is retrieved?**

This is the interesting one. If early-stage and severe-stage queries return identical
chunks, then the severity signal is not doing any work in retrieval and only matters at
prompt time. Either result is worth knowing and reporting.

In [ ]:
from app.rag.retriever import retrieve_evidence
from app.schemas.diagnosis import ClassPrediction, Diagnosis, SeverityReport
from app.schemas.environment import EnvironmentContext, SoilType
from app.vision.severity import classify_tier

diagnosis = Diagnosis(
    top=ClassPrediction(label="Tomato___Late_blight", crop="Tomato",
                        disease="Late blight", confidence=0.93),
    checkpoint_loaded=True, confidence_reliable=True,
)


def severity_at(pct: float) -> SeverityReport:
    return SeverityReport(
        severity_pct=pct, necrotic_pct=pct * 0.6, chlorotic_pct=pct * 0.4,
        healthy_pct=100 - pct, lesion_count=6, largest_lesion_pct=pct / 4,
        leaf_coverage_pct=42.0, action_tier=classify_tier(pct),
    )


environment = EnvironmentContext(region="Nashik", soil_type=SoilType.BLACK,
                                 rainfall_last_7d_mm=110)

retrieved_sets = {}
for pct in (3.0, 18.0, 45.0):
    severity = severity_at(pct)
    evidence, query = retrieve_evidence(knowledge, diagnosis, severity, environment, settings)
    retrieved_sets[severity.action_tier.value] = [item.text[:80] for item in evidence]

    print(f"\n=== severity {pct:.0f} % -> {severity.action_tier.value.upper()} ===")
    print(f"query: {query}")
    for item in evidence:
        print(f"  [{item.id}] {item.score:.3f}  {item.source[:55]}")

tiers = list(retrieved_sets)
for i in range(len(tiers) - 1):
    a, b = set(retrieved_sets[tiers[i]]), set(retrieved_sets[tiers[i + 1]])
    overlap = len(a & b) / max(len(a | b), 1)
    print(f"\noverlap {tiers[i]} vs {tiers[i + 1]}: {overlap:.0%}")

## 5. Switching to pgvector

```powershell
docker compose up -d postgres
```

Then set `DATABASE_URL` in `.env` and re-ingest. The init SQL in
`infra/postgres/init/01_pgvector.sql` creates the `vector` extension plus the provenance
and advisory-log tables; LlamaIndex creates its own `data_<table>` with an HNSW cosine index
on first write.

In [ ]:
from app.rag.index import try_pgvector_store

if not settings.database_url:
    print("DATABASE_URL is unset — set it in .env to try pgvector.")
else:
    store = try_pgvector_store(settings, settings.embedding_dim)
    if store is None:
        print("Postgres is not reachable. Start it with: docker compose up -d postgres")
    else:
        print(f"Connected. Re-ingesting into table data_{settings.pgvector_table} ...")
        pg_knowledge, pg_report = ingest_knowledge(settings, prefer_pgvector=True, reset=True)
        print(pg_report)
        for node in pg_knowledge.index.as_retriever(similarity_top_k=3).retrieve(
            "late blight severe intervention"
        ):
            print(f"  {node.score:.3f}  {node.node.metadata.get('file_name')}")

## 6. Adding your own documents

1. Drop a `.md`, `.txt` or `.pdf` into `data/knowledge/`.
2. For Markdown, add front matter — `crop`, `disease`, `region`, `doc_type`, `source`.
   `source` is what the citation displays, so make it something a farmer could look up.
3. Re-ingest: `python scripts/ingest_knowledge.py`, or `POST /api/v1/knowledge/ingest`.

Prefer many small focused documents over a few large ones: retrieval granularity is bounded
by document boundaries as much as by chunk size, and a bulletin covering one disease on one
crop retrieves far more precisely than a 40-page compendium.

In [ ]:
counts = {}
for document in documents:
    key = (document.metadata["crop"], document.metadata["disease"])
    counts[key] = counts.get(key, 0) + 1

print("CURRENT COVERAGE")
for (crop, disease), count in sorted(counts.items()):
    print(f"  {crop:<12} {disease:<16} {count}")

print("\nGAPS WORTH FILLING NEXT (from the 38 PlantVillage classes)")
from app.vision.labels import PLANTVILLAGE_CLASSES, is_healthy, split_label

covered = {(crop.lower(), disease.lower()) for crop, disease in counts}
missing = [
    f"{crop} / {disease}"
    for crop, disease in (split_label(label) for label in PLANTVILLAGE_CLASSES)
    if not is_healthy(f"{crop}___{disease}") and (crop.lower(), disease.lower()) not in covered
]
for entry in missing[:12]:
    print(f"  - {entry}")
print(f"  ... {len(missing)} classes have no dedicated document yet")